# MathWriting Data Pipeline

Converts the MathWriting InkML dataset into model-ready inputs:
1. **PNGs** — each ink rescaled to a fixed height (aspect ratio preserved, no letterboxing/squashing to a square), rendered as grayscale line art.
2. **Tokenized labels** — each `normalizedLabel` (or `label` for `symbols/`, which has no normalized version) split into LaTeX tokens and mapped to integer ids via a frozen vocabulary.
3. **A calibrated augmentation utility** — rotation, shear, stroke thinning, and Gaussian blur, each independently randomized and composable, ready to call from a training-time `Dataset`.

Pipeline stages, in order:
- Download the dataset if it isn't already present locally (never committed to the repo)
- Read InkML files (strokes + annotations)
- Geometrically normalize each ink (fixed height, preserved aspect ratio)
- Rasterize to PNG (Cairo if available, otherwise a Pillow fallback)
- Tokenize labels with the official regex tokenizer
- Build a vocabulary from `train` + `synthetic` only (never from `valid`/`test`)
- Write per-split `.jsonl` label files + `vocab.json` + `metadata.json`
- Run sanity checks (tokenizer round-trip, split disjointness, distribution stats)
- Spot-check a few rendered images against their labels
- Define + visually QA the augmentation utility used at training time

`train`/`valid`/`test`/`synthetic`/`symbols` are processed independently and never merged — the official splits already account for writer and expression overlap, so we preserve them exactly as given.

**On augmentation:** `processed/images/` holds only *clean* (unaugmented) renders for every split, including `train`. Augmentation is intentionally **not** baked into these PNGs — it's applied online, at training time, by calling `render_with_augmentation` fresh for each `train` example on each epoch (see the dedicated section below). This gives unbounded diversity across epochs instead of a fixed set of pre-baked copies, and keeps `processed/` simple: one clean image per sample, always.

## Setup

Detects whether `pycairo` is usable and installs any of the small set of dependencies that aren't already available. Cairo gives higher-fidelity anti-aliased rendering (matching the official example notebook), but its system library is often unavailable on Windows, so we fall back to a supersampled Pillow renderer instead of failing.

In [ ]:
try:
    import PIL, numpy, matplotlib  # noqa: F401
except ImportError:
    %pip install -q pillow numpy matplotlib

try:
    import cairo  # noqa: F401
    _CAIRO_AVAILABLE = True
except ImportError:
    try:
        %pip install -q pycairo
        import cairo  # noqa: F401
        _CAIRO_AVAILABLE = True
    except Exception as e:
        print(f"pycairo unavailable ({e!r}); using the Pillow fallback renderer instead.")
        _CAIRO_AVAILABLE = False

print(f"Using {'Cairo' if _CAIRO_AVAILABLE else 'Pillow (supersampled)'} for rendering.")

## Download the dataset

The dataset is a public Google Cloud Storage bucket, not something we commit to the repo (see the project `.gitignore`). This step is idempotent — it only downloads/extracts if `mathwriting-2024-excerpt/` isn't already present locally, so re-running this notebook never re-downloads.

To use the full dataset instead of the excerpt, change `DATASET_URL` below to `.../mathwriting-2024.tgz` and update `ROOT_DIR` to match.

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

DATASET_URL = "https://storage.googleapis.com/mathwriting_data/mathwriting-2024-excerpt.tgz"
ROOT_DIR = Path("mathwriting-2024-excerpt")

if not ROOT_DIR.exists():
    archive_name = Path(DATASET_URL).name
    print(f"'{ROOT_DIR}' not found locally; downloading from {DATASET_URL} ...")
    urllib.request.urlretrieve(DATASET_URL, archive_name)
    with tarfile.open(archive_name) as tar:
        tar.extractall(".")
    print(f"Extracted to '{ROOT_DIR}'.")
else:
    print(f"'{ROOT_DIR}' already present, skipping download.")

## Imports & configuration

All the tunable constants live here. Notes on a few choices:
- `TARGET_HEIGHT=64`: based on measuring this excerpt's actual ink aspect ratios (median ~3.2:1, p99 ~6.5:1), this keeps p99 widths around ~420px — a reasonable size for a MobileNetV3 encoder.
- Per-image widths are **not** padded or rounded here — we store each image's true (unpadded) width in the label files, and defer batch padding + the padding mask to the training-time collate function, where the batch's max width gets rounded up to a multiple of 32 (MobileNetV3's stride).
- `VOCAB_SPLITS = ["train", "synthetic"]`: the vocabulary is built only from data intended for training, per the dataset README's own guidance. `valid`/`test` tokens never influence the vocab — unseen tokens there become `<UNK>`, which is itself a useful generalization signal.

In [ ]:
import re
import math
import json
import itertools
import dataclasses
from collections import Counter
from xml.etree import ElementTree

import numpy as np
from PIL import Image, ImageDraw, ImageFilter
import matplotlib.pyplot as plt

# --- Paths ---
OUTPUT_DIR = Path("processed")

# `symbols/` has no `normalizedLabel` (only `label`); the rest have both.
ALL_SPLITS = ["train", "valid", "test", "synthetic", "symbols"]
VOCAB_SPLITS = ["train", "synthetic"]

# --- Image normalization ---
TARGET_HEIGHT = 64        # pixels, after geometric normalization
STROKE_WIDTH_PX = 2.5     # constant rendered stroke width, in pixels
MARGIN_PX = 4             # blank margin added around the ink on each side
SUPERSAMPLE = 4           # Pillow-fallback (and stroke-thinning) supersample factor

# --- Special tokens ---
PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN = "<PAD>", "<BOS>", "<EOS>", "<UNK>"
SPECIAL_TOKENS = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]

for split in ALL_SPLITS:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "labels").mkdir(parents=True, exist_ok=True)

## Reading InkML files

Adapted directly from the official example notebook: each file's `<trace>` elements become stroke arrays of shape `(2, num_points)` (x, y — we drop the timestamp, which this pipeline doesn't use), and `<annotation>` elements become the metadata dict (`label`, `normalizedLabel`, `sampleId`, etc.).

In [ ]:
@dataclasses.dataclass
class Ink:
    """A single ink: its strokes and InkML annotations."""
    strokes: list  # each element is an array of shape (2, num_points): rows are (x, y)
    annotations: dict


def read_inkml_file(filename) -> Ink:
    """Parses a single MathWriting InkML file."""
    with open(filename, "r", encoding="utf-8") as f:
        root = ElementTree.fromstring(f.read())

    strokes = []
    annotations = {}

    for element in root:
        tag_name = element.tag.split("}")[-1]
        if tag_name == "annotation":
            annotations[element.attrib.get("type")] = element.text
        elif tag_name == "trace":
            xs, ys = [], []
            for point in element.text.split(","):
                x, y, _t = point.split(" ")
                xs.append(float(x))
                ys.append(float(y))
            strokes.append(np.array((xs, ys)))

    return Ink(strokes=strokes, annotations=annotations)

## Geometric normalization

Raw InkML coordinates are unnormalized (readme: "vertical extent of a given letter can vary greatly depending on the ink and writer"). We rescale each ink **uniformly** (same factor for x and y, so nothing gets stretched or squashed) so its bounding-box height equals `TARGET_HEIGHT`, then shift it into a small margin. Width is whatever falls out of that — we deliberately don't force a square or a fixed canvas.

This is split into two composable pieces — `rescale_to_height` (pure scale-to-`TARGET_HEIGHT`) and `fit_canvas` (shift into the margin + measure canvas size) — so augmentation (below) can slot in *between* them: normalize height first, jitter geometry second, then re-fit the canvas to the jittered result.

In [ ]:
def rescale_to_height(ink: Ink, target_height: float = TARGET_HEIGHT) -> Ink:
    """Uniformly scales (x and y by the same factor) so the ink's bounding-box
    height equals target_height. Does not reposition the ink."""
    ys = np.concatenate([s[1] for s in ink.strokes])
    ink_height = max(ys.max() - ys.min(), 1e-6)  # guards against a single-point/flat ink
    scale = target_height / ink_height
    return Ink(strokes=[s * scale for s in ink.strokes], annotations=ink.annotations)


def fit_canvas(ink: Ink, margin: int = MARGIN_PX):
    """Shifts the ink so its bounding box starts at (margin, margin), and
    measures the canvas size that snugly fits it. Recomputed from the actual
    bounding box (not assumed), so this works after rotation/shear have
    changed the ink's extent.

    Returns (shifted_ink, canvas_width, canvas_height).
    """
    xs = np.concatenate([s[0] for s in ink.strokes])
    ys = np.concatenate([s[1] for s in ink.strokes])
    x_min, y_min = xs.min(), ys.min()

    new_strokes = [
        np.stack([s[0] - x_min + margin, s[1] - y_min + margin])
        for s in ink.strokes
    ]

    width = int(math.ceil(max(s[0].max() for s in new_strokes) + margin))
    height = int(math.ceil(max(s[1].max() for s in new_strokes) + margin))

    return Ink(strokes=new_strokes, annotations=ink.annotations), width, height

## Rendering

Two renderers with the same signature, dispatched on `_CAIRO_AVAILABLE`:
- **Cairo** (`_render_with_cairo`): same approach as the official example notebook — round line caps/joins, natively anti-aliased.
- **Pillow fallback** (`_render_with_pillow`): draws at `SUPERSAMPLE`× resolution (a disk at every point plus a curved polyline, to approximate round caps/joins) then downsamples with Lanczos — a standard way to fake anti-aliasing without any system library dependency.

Both output a single-channel (`"L"`) grayscale image: black strokes on a white background, no color information to carry.

In [ ]:
def _render_with_cairo(ink: Ink, width: int, height: int, stroke_width: float) -> Image.Image:
    surface = cairo.ImageSurface(cairo.FORMAT_ARGB32, width, height)
    ctx = cairo.Context(surface)
    ctx.set_source_rgb(1, 1, 1)
    ctx.paint()
    ctx.set_source_rgb(0, 0, 0)
    ctx.set_line_width(stroke_width)
    ctx.set_line_cap(cairo.LineCap.ROUND)
    ctx.set_line_join(cairo.LineJoin.ROUND)

    for stroke in ink.strokes:
        if stroke.shape[1] == 1:
            x, y = stroke[0, 0], stroke[1, 0]
            ctx.arc(x, y, stroke_width / 2, 0, 2 * math.pi)
            ctx.fill()
        else:
            ctx.move_to(stroke[0, 0], stroke[1, 0])
            for x, y in stroke[:, 1:].T:
                ctx.line_to(x, y)
            ctx.stroke()

    size = (surface.get_width(), surface.get_height())
    stride = surface.get_stride()
    with surface.get_data() as memory:
        rgb_image = Image.frombuffer("RGB", size, memory.tobytes(), "raw", "BGRX", stride)
    return rgb_image.convert("L")


def _render_with_pillow(ink: Ink, width: int, height: int, stroke_width: float,
                         supersample: int = SUPERSAMPLE) -> Image.Image:
    """Anti-alias-by-supersampling fallback; needs no system libraries."""
    big = Image.new("L", (width * supersample, height * supersample), color=255)
    draw = ImageDraw.Draw(big)
    line_width = max(1, round(stroke_width * supersample))
    radius = line_width / 2

    for stroke in ink.strokes:
        pts = list(zip(stroke[0] * supersample, stroke[1] * supersample))
        if len(pts) == 1:
            x, y = pts[0]
            draw.ellipse([x - radius, y - radius, x + radius, y + radius], fill=0)
        else:
            draw.line(pts, fill=0, width=line_width, joint="curve")
            # `joint="curve"` rounds interior joints but not the two end caps,
            # so cap every point with a small disk to emulate Cairo's round caps.
            for x, y in pts:
                draw.ellipse([x - radius, y - radius, x + radius, y + radius], fill=0)

    return big.resize((width, height), Image.LANCZOS)


def render_ink(ink: Ink, width: int, height: int, *, stroke_width: float = STROKE_WIDTH_PX) -> Image.Image:
    if _CAIRO_AVAILABLE:
        return _render_with_cairo(ink, width, height, stroke_width)
    return _render_with_pillow(ink, width, height, stroke_width)

## Augmentation

- `apply_affine_ink` rotates/shears strokes around the ink's own bounding-box center — a single 2×2 linear map. This runs on vector stroke coordinates, then the result is re-rendered from scratch, which avoids the blur/interpolation artifacts of warping a finished raster image.
- `thin_strokes` erodes strokes by re-rendering at supersampled resolution, running a 3×3 max-filter pass per iteration there, then downsampling back down. Eroding directly at the final ~64px resolution is too coarse: a single max-filter pass removes close to a full pixel of a ~2.5px-wide stroke, wiping it out after 2-3 iterations. Supersampling first makes each iteration a much finer, sub-pixel-equivalent nibble.
- `gaussian_blur` is a plain `ImageFilter.GaussianBlur` on the rendered image.

In [ ]:
def apply_affine_ink(ink: Ink, *, angle_deg: float = 0.0, shear_deg: float = 0.0) -> Ink:
    """Rotates/shears strokes around the ink's own bounding-box center."""
    xs = np.concatenate([s[0] for s in ink.strokes])
    ys = np.concatenate([s[1] for s in ink.strokes])
    center = np.array([[(xs.min() + xs.max()) / 2], [(ys.min() + ys.max()) / 2]])

    theta = math.radians(angle_deg)
    shear = math.radians(shear_deg)
    rotation = np.array([[math.cos(theta), -math.sin(theta)],
                         [math.sin(theta), math.cos(theta)]])
    shear_mat = np.array([[1.0, math.tan(shear)],
                          [0.0, 1.0]])
    transform = rotation @ shear_mat

    new_strokes = [transform @ (s - center) + center for s in ink.strokes]
    return Ink(strokes=new_strokes, annotations=ink.annotations)


def thin_strokes(canvas_ink: Ink, width: int, height: int, iterations: int,
                  supersample: int = SUPERSAMPLE, stroke_width: float = STROKE_WIDTH_PX) -> Image.Image:
    """Erodes strokes via supersampled max-filter passes (see markdown above for why)."""
    hi_res_ink = Ink(
        strokes=[s * supersample for s in canvas_ink.strokes],
        annotations=canvas_ink.annotations,
    )
    hi_res_image = render_ink(hi_res_ink, width * supersample, height * supersample,
                               stroke_width=stroke_width * supersample)
    for _ in range(iterations):
        hi_res_image = hi_res_image.filter(ImageFilter.MaxFilter(3))
    return hi_res_image.resize((width, height), Image.LANCZOS)


def gaussian_blur(image: Image.Image, radius: float) -> Image.Image:
    return image.filter(ImageFilter.GaussianBlur(radius))

## Augmentation: policy

Each transform rolls **independently** with its own probability, and any subset can stack on a given sample — there's no "pick exactly one" or "always apply all." Ranges come straight from calibration:

| transform | mechanism | range | probability |
|---|---|---|---|
| rotation | ink-level affine | ±6° | 0.5 |
| shear | ink-level affine | ±8° | 0.4 |
| thinning | pixel-space erosion | 1–3 iterations (int) | 0.3 |
| blur | pixel-space Gaussian | radius 0.3–1.0 | 0.3 |

Rotation/shear are symmetric around 0, so a continuous draw from their range already includes near-zero outcomes — `prob` mainly controls how often a *given* sample moves away from that. Thinning/blur have no "off" value inside their calibrated range at all (there's no such thing as -1 blur), so for those two `prob` alone decides clean-vs-degraded for that sample.

**This is meant to be called at training time**, once per `train` example per epoch (e.g. from a `Dataset.__getitem__`), passing a fresh `np.random.Generator` draw each call — not run once here to bake a fixed augmented copy to disk. That's why `processed/images/train/` (built further down) stays clean: it's the frozen geometry+label source a training `Dataset` reads from, and this function is what that `Dataset` calls on top of it.

In [ ]:
RNG_SEED = 0

AUGMENTATION_CONFIG = {
    "rotation_deg":   {"prob": 0.5, "range": (-6, 6),    "dtype": float},
    "shear_deg":      {"prob": 0.4, "range": (-8, 8),    "dtype": float},
    "thinning_iters": {"prob": 0.3, "range": (1, 3),     "dtype": int},
    "blur_radius":    {"prob": 0.3, "range": (0.3, 1.0), "dtype": float},
}


def _roll(cfg: dict, rng: np.random.Generator):
    """Returns a sampled parameter value if this transform fires this draw, else None."""
    if rng.random() >= cfg["prob"]:
        return None
    low, high = cfg["range"]
    value = rng.uniform(low, high)
    return int(round(value)) if cfg["dtype"] is int else value


def render_with_augmentation(normalized_ink: Ink, rng: np.random.Generator):
    """Renders one (already height-normalized) ink with a fresh random draw of
    the configured augmentations. Returns (image, width, height, applied), where
    `applied` records which transforms fired and with what parameter — useful
    for debugging or logging what a given training step actually saw."""
    applied = {}
    ink = normalized_ink

    angle = _roll(AUGMENTATION_CONFIG["rotation_deg"], rng)
    shear = _roll(AUGMENTATION_CONFIG["shear_deg"], rng)
    if angle is not None or shear is not None:
        ink = apply_affine_ink(ink, angle_deg=angle or 0.0, shear_deg=shear or 0.0)
        if angle is not None:
            applied["rotation_deg"] = angle
        if shear is not None:
            applied["shear_deg"] = shear

    canvas_ink, width, height = fit_canvas(ink)
    image = render_ink(canvas_ink, width, height)

    thinning_iters = _roll(AUGMENTATION_CONFIG["thinning_iters"], rng)
    if thinning_iters is not None:
        image = thin_strokes(canvas_ink, width, height, iterations=thinning_iters)
        applied["thinning_iters"] = thinning_iters

    blur_radius = _roll(AUGMENTATION_CONFIG["blur_radius"], rng)
    if blur_radius is not None:
        image = gaussian_blur(image, blur_radius)
        applied["blur_radius"] = blur_radius

    return image, width, height, applied

## Tokenizer

The official regex tokenizer, unchanged: splits a LaTeX string into meaningful units (`\frac`, `\alpha`, single characters, etc.) rather than raw characters.

In [ ]:
_COMMAND_RE = re.compile(r'\\(mathbb{[a-zA-Z]}|begin{[a-z]+}|end{[a-z]+}|operatorname\*|[a-zA-Z]+|.)')


def tokenize_expression(s: str) -> list:
    r"""Splits a LaTeX math string into tokens, e.g. r'\frac{1}{2}' ->
    ['\\frac', '{', '1', '}', '{', '2', '}']."""
    tokens = []
    while s:
        if s[0] == "\\":
            tokens.append(_COMMAND_RE.match(s).group(0))
        else:
            tokens.append(s[0])
        s = s[len(tokens[-1]):]
    return tokens


def get_label_text(annotations: dict) -> str:
    """Prefer normalizedLabel; symbols/ inks only provide 'label'."""
    return annotations.get("normalizedLabel") or annotations["label"]

## Per-split processing

For every InkML file in a split: read it, normalize+render it to `processed/images/{split}/{sample_id}.png`, tokenize its label, and collect a record. This produces the **clean** (unaugmented) baseline for every split, including `train` — augmentation is applied later, on top of this, at training time (see above). Vocabulary lookup happens later, in its own cell — this function only tokenizes, it doesn't assign ids yet.

In [ ]:
def process_split(split: str) -> list:
    split_dir = ROOT_DIR / split
    image_dir = OUTPUT_DIR / "images" / split
    records = []

    filenames = sorted(p for p in split_dir.iterdir() if p.suffix == ".inkml")
    for path in filenames:
        sample_id = path.stem
        ink = read_inkml_file(path)

        normalized_ink = rescale_to_height(ink)
        canvas_ink, width, height = fit_canvas(normalized_ink)
        image = render_ink(canvas_ink, width, height)
        image.save(image_dir / f"{sample_id}.png")

        label = get_label_text(ink.annotations)
        tokens = tokenize_expression(label)

        records.append({
            "sample_id": sample_id,
            "split": split,
            "label": ink.annotations.get("label"),
            "normalized_label": ink.annotations.get("normalizedLabel"),  # None for symbols/
            "tokens": tokens,
            "width": width,
            "height": height,
            "num_strokes": len(normalized_ink.strokes),
        })

    return records

## Run processing over all splits

This is the actual (potentially slow) work — reading every InkML file and rendering every PNG. `train`/`valid`/`test`/`synthetic`/`symbols` are each processed independently into their own output subfolder; nothing here mixes them.

In [ ]:
records_by_split = {}
for split in ALL_SPLITS:
    print(f"Processing '{split}'...")
    records_by_split[split] = process_split(split)
    print(f"  {len(records_by_split[split])} inks -> {OUTPUT_DIR / 'images' / split}")

## Vocabulary construction

Built only from `VOCAB_SPLITS` (`train` + `synthetic`). Token ids: special tokens first (`<PAD>=0, <BOS>=1, <EOS>=2, <UNK>=3`), then every observed token ordered by frequency (most common first, alphabetical tiebreak) for a deterministic, reproducible mapping.

In [ ]:
def build_vocab(splits: list) -> dict:
    counts = Counter()
    for split in splits:
        for rec in records_by_split[split]:
            counts.update(rec["tokens"])

    ordered_tokens = [tok for tok, _ in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))]

    vocab = {tok: idx for idx, tok in enumerate(SPECIAL_TOKENS)}
    for tok in ordered_tokens:
        vocab[tok] = len(vocab)

    return vocab


vocab = build_vocab(VOCAB_SPLITS)
print(f"Vocab size (built from {VOCAB_SPLITS}): {len(vocab)}")

with open(OUTPUT_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

## Token ids + label files

Maps each record's tokens to ids (unknown tokens -> `<UNK>`), wraps the sequence in `<BOS>`/`<EOS>`, and writes one `.jsonl` per split. Also reports each split's out-of-vocabulary rate — this should be ~0% for `train`/`synthetic` (they defined the vocab) and is a meaningful diagnostic for `valid`/`test`/`symbols`.

In [ ]:
def tokens_to_ids(tokens: list, vocab: dict) -> list:
    unk_id = vocab[UNK_TOKEN]
    return [vocab.get(tok, unk_id) for tok in tokens]


for split in ALL_SPLITS:
    n_unk = 0
    n_tok = 0
    out_path = OUTPUT_DIR / "labels" / f"{split}.jsonl"

    with open(out_path, "w", encoding="utf-8") as f:
        for rec in records_by_split[split]:
            ids = tokens_to_ids(rec["tokens"], vocab)
            n_unk += sum(1 for i in ids if i == vocab[UNK_TOKEN])
            n_tok += len(ids)

            rec_out = dict(rec)
            rec_out["token_ids"] = [vocab[BOS_TOKEN]] + ids + [vocab[EOS_TOKEN]]
            f.write(json.dumps(rec_out, ensure_ascii=False) + "\n")

    oov_rate = (n_unk / n_tok) if n_tok else 0.0
    print(f"{split:10s} wrote {len(records_by_split[split]):4d} records -> {out_path}  (OOV rate {oov_rate:.2%})")

## Metadata

Records the config this run used, so the training code (and future-you) doesn't have to guess — including the augmentation policy (for documentation, since it isn't baked into the PNGs) and an explicit reminder that width padding/masking is a training-time concern.

In [ ]:
metadata = {
    "target_height_px": TARGET_HEIGHT,
    "stroke_width_px": STROKE_WIDTH_PX,
    "margin_px": MARGIN_PX,
    "renderer": "cairo" if _CAIRO_AVAILABLE else "pillow-supersampled",
    "supersample_factor": None if _CAIRO_AVAILABLE else SUPERSAMPLE,
    "width_padding_note": (
        "Per-image widths are stored as-is, not padded or rounded. At batch "
        "collate time: pad every image in the batch up to the batch's max "
        "width (rounded up to a multiple of 32, MobileNetV3's stride), and "
        "build a padding mask from each sample's true 'width' field so the "
        "decoder's cross-attention ignores the padded columns."
    ),
    "augmentation_note": (
        "processed/images/ contains only CLEAN renders for every split. "
        "Augmentation (see AUGMENTATION_CONFIG / render_with_augmentation in "
        "this notebook) is applied online at training time to `train` examples "
        "only, re-sampled fresh every epoch -- it is not baked into these PNGs."
    ),
    "augmentation_config": {
        name: {**cfg, "dtype": cfg["dtype"].__name__} for name, cfg in AUGMENTATION_CONFIG.items()
    },
    "special_tokens": {tok: vocab[tok] for tok in SPECIAL_TOKENS},
    "vocab_size": len(vocab),
    "vocab_built_from_splits": VOCAB_SPLITS,
    "all_splits_processed": ALL_SPLITS,
}

with open(OUTPUT_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

metadata

## Sanity checks

Three checks before trusting the output:
1. **Tokenizer round-trip** — joining a record's tokens back together must exactly reproduce its source label. A failure here means the tokenizer regex silently dropped/misparsed a character.
2. **Split disjointness** — `train`/`valid`/`test`/`synthetic`/`symbols` must never share a `sample_id`. This should always hold (they're separate directories) but it's cheap to confirm rather than assume.
3. **Distribution stats** — width/token-length percentiles, to confirm `TARGET_HEIGHT=64` was a reasonable choice for this data (it was tuned on this excerpt, and the full dataset may differ).

In [ ]:
# 1. Tokenizer round-trip.
roundtrip_failures = []
for split in ALL_SPLITS:
    for rec in records_by_split[split]:
        label = rec["normalized_label"] or rec["label"]
        if "".join(rec["tokens"]) != label:
            roundtrip_failures.append((split, rec["sample_id"]))

print(f"Tokenizer round-trip failures: {len(roundtrip_failures)}")
if roundtrip_failures:
    print(roundtrip_failures[:10])

# 2. Split disjointness.
id_sets = {split: {r["sample_id"] for r in records_by_split[split]} for split in ALL_SPLITS}
for a, b in itertools.combinations(ALL_SPLITS, 2):
    overlap = id_sets[a] & id_sets[b]
    if overlap:
        print(f"WARNING: '{a}' and '{b}' share {len(overlap)} sample id(s): {list(overlap)[:5]}")
assert all(not (id_sets[a] & id_sets[b]) for a, b in itertools.combinations(ALL_SPLITS, 2)), \
    "Splits are supposed to be disjoint -- something is very wrong."
print("All splits confirmed disjoint.")

# 3. Distribution stats.
print()
for split in ["train", "valid", "test"]:
    widths = np.array([r["width"] for r in records_by_split[split]])
    tok_lens = np.array([len(r["tokens"]) for r in records_by_split[split]])
    print(
        f"{split:10s} n={len(widths):4d}  "
        f"width px: median={np.median(widths):.0f} p99={np.percentile(widths, 99):.0f} max={widths.max():.0f}  |  "
        f"tokens: median={np.median(tok_lens):.0f} p99={np.percentile(tok_lens, 99):.0f} max={tok_lens.max()}"
    )

## Visual spot-check

Renders a handful of `train` PNGs next to their labels straight off disk — the fastest way to catch a rendering bug (e.g. a wrong scale factor or a botched stroke join) that stats alone wouldn't reveal.

In [ ]:
sample_records = records_by_split["train"][:4]
fig, axes = plt.subplots(1, len(sample_records), figsize=(16, 4))

for ax, rec in zip(axes, sample_records):
    img_path = OUTPUT_DIR / "images" / "train" / f"{rec['sample_id']}.png"
    ax.imshow(Image.open(img_path), cmap="gray")
    ax.set_title(rec["normalized_label"], fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

## Augmentation QA

Visually validates the finalized augmentation policy on real `train` samples, saved separately to `augmented_demo/` (kept apart from `processed/`, which must stay clean). For each of a few samples we show the clean baseline plus 3 **independent fresh draws** from `render_with_augmentation` — i.e. exactly what a training `Dataset` would produce across different epochs — to eyeball whether stacked transforms still look like legible handwriting rather than isolating each transform in a vacuum.

In [ ]:
AUG_QA_DIR = Path("augmented_demo")
AUG_QA_DIR.mkdir(parents=True, exist_ok=True)

qa_rng = np.random.default_rng(RNG_SEED)
qa_sample_ids = [rec["sample_id"] for rec in records_by_split["train"][:4]]
n_draws_per_sample = 3

qa_records = []
for sample_id in qa_sample_ids:
    ink = read_inkml_file(ROOT_DIR / "train" / f"{sample_id}.inkml")
    normalized_ink = rescale_to_height(ink)

    base_canvas_ink, base_width, base_height = fit_canvas(normalized_ink)
    base_image = render_ink(base_canvas_ink, base_width, base_height)
    base_path = AUG_QA_DIR / f"{sample_id}_clean.png"
    base_image.save(base_path)
    qa_records.append({"label": f"{sample_id}\nclean", "path": base_path})

    for draw_idx in range(n_draws_per_sample):
        image, width, height, applied = render_with_augmentation(normalized_ink, qa_rng)
        path = AUG_QA_DIR / f"{sample_id}_draw{draw_idx}.png"
        image.save(path)
        applied_str = ", ".join(f"{k}={v:.2g}" if isinstance(v, float) else f"{k}={v}"
                                 for k, v in applied.items()) or "none"
        qa_records.append({"label": f"draw {draw_idx}\n{applied_str}", "path": path})

n_cols = n_draws_per_sample + 1
n_rows = len(qa_sample_ids)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))

for row, sample_id in enumerate(qa_sample_ids):
    row_records = qa_records[row * n_cols:(row + 1) * n_cols]
    for ax, rec in zip(axes[row], row_records):
        ax.imshow(Image.open(rec["path"]), cmap="gray")
        ax.set_title(rec["label"], fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()